# **Anomaly Detection Model Training Notebook**



---
## Setup Environment

In [2]:
# DO NOT MODIFY THE CODE IN THIS CELL
!pip install -q utstd

from utstd.folders import *
from utstd.ipyrenders import *

at = AtFolder(
    course_code=36106,
    assignment="AT3",
)
at.run()

import warnings
warnings.simplefilter(action='ignore')

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
umap-learn 0.5.12 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
hdbscan 0.8.42 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).

You can now save your data files in: /content/gdrive/MyDrive/36106/assignment/AT3/data


---
## Student Information

In [3]:
# <Student to fill this section>
group_name = "Neural Minds"
student_name = "Aryaman Malik"
student_id = "25980501"

In [4]:
# Do not modify this code
print_tile(size="h1", key='group_name', value=group_name)

In [5]:
# Do not modify this code
print_tile(size="h1", key='student_name', value=student_name)

In [6]:
# Do not modify this code
print_tile(size="h1", key='student_id', value=student_id)

---
## 0. Python Packages

### 0.a Install Additional Packages

> If you are using additional packages, you need to install them here using the command: `! pip install <package_name>`

In [7]:
# <Student to fill this section>
!pip install -q scikit-learn
!pip install -q "vegafusion[embed]>=1.5.0"
!pip install -q "vl-convert-python>=1.6.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 26.0 MB/s eta 0:00:00


### 0.b Import Packages

In [8]:
# <Student to fill this section>
import pandas as pd
import altair as alt

In [9]:
import numpy as np
from itertools import product
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (precision_score, recall_score, f1_score,
                              roc_auc_score, average_precision_score,
                              confusion_matrix, precision_recall_curve)
# Enable vegafusion for Altair large dataset support
alt.data_transformers.enable("vegafusion")

DataTransformerRegistry.enable('vegafusion')

---
## B. Business Understanding

In [10]:
# <Student to fill this section>
business_use_case_description = """
A global cycling-equipment retailer has accumulated 42,100 sales-order line items across three years of trading but has no automated mechanism to detect unusual transactions. Manual audit relies on random sampling, which surfaces only ~6 real anomalies per 100 lines reviewed and biases reviewer time toward recent or large-value lines.

We frame this as an UNSUPERVISED ANOMALY DETECTION task. The model assigns every line item an anomaly score; lines with high scores are flagged for review by the revenue-assurance team. Three operational patterns the business cares about:

1. Lines sold at unusually deep discounts (pricing errors or unauthorised markdowns).
2. Lines with abnormally large order quantities (clerical errors or unusual bulk orders).
3. Offline reseller lines priced inconsistently with that product's typical wholesale rate.

The model serves as a triage tool: an auditor reviews the top-N highest-scoring lines from each daily batch rather than sampling randomly.
"""


In [11]:
# Do not modify this code
print_tile(size="h3", key='business_use_case_description', value=business_use_case_description)

In [12]:
# <Student to fill this section>
business_objectives = """
Primary objective: ENABLE AUDITOR-EFFICIENT TRANSACTION REVIEW.
Auditors should be able to find at least 5x more real anomalies per hour of review by working through model-ranked lines instead of random samples.

Secondary objectives:
1. Achieve at least 90% recall on financially material anomalies (offline pricing irregularities and large bulk-order outliers), measured in both line count and total revenue exposure.
2. Maintain precision >= 30% at the chosen operating threshold so the auditor team is not overwhelmed by false alarms.
3. Score new lines in under 5 seconds per batch of 1,000 for real-time deployment.
4. Provide interpretable feature contributions so auditors can act on flags with context.

IMPACT OF CORRECT VS INCORRECT RESULTS
- True positive: an auditor catches a real pricing error or unusual bulk order before it costs revenue. Direct dollar savings.
- False positive: an auditor wastes ~15 minutes reviewing a legitimate transaction. Acceptable up to a precision floor of ~30% (one wasted review per two real catches).
- False negative: a real anomaly slips through. Cost varies dramatically with line value - a USD 5,000 unauthorised bike discount is far more material than a USD 5 over-discounted cap.
- True negative: a normal line correctly not flagged. The dominant case (94% of lines) and the silent success of the system.
"""

In [13]:
# Do not modify this code
print_tile(size="h3", key='business_objectives', value=business_objectives)

In [14]:
# <Student to fill this section>
stakeholders_expectations_explanations = """
Three stakeholder groups have a direct interest in the model:

REVENUE-ASSURANCE / AUDIT TEAM (primary user)
- Expects ranked daily report of top-50 anomalous lines.
- Needs feature explanations alongside flags (which features drove this score?).
- Will provide feedback (true anomaly / false positive) to retrain the model.

PRICING & SALES OPERATIONS (decision authority)
- Expects to use confirmed-anomaly lists to renegotiate discount policy with under-performing resellers.
- Wants quarterly summaries of detected pattern types (online vs offline; small-value vs material).

EXECUTIVE & COMPLIANCE (downstream stakeholder)
- Expects clear documentation of model limitations and ethical guardrails.
- Wants assurance that the model assists rather than automates audit decisions, and that no customer-level or territory-level discrimination is encoded.
- Approves only model deployments that include explicit threshold-tuning controls and a feedback loop.
"""

In [15]:
# Do not modify this code
print_tile(size="h3", key='stakeholders_expectations_explanations', value=stakeholders_expectations_explanations)

---
## C. Data Understanding

### C.1   Load Datasets


In [16]:
# <Student to fill this section>
product_category_df = pd.read_csv(at.folder_path / "product_category.csv")
product_sub_category_df = pd.read_csv(at.folder_path / "product_sub_category.csv")
product_df = pd.read_csv(at.folder_path / "product.csv")
sales_order_detail_df = pd.read_csv(at.folder_path / "sales_order_detail.csv")
sales_order_header_df = pd.read_csv(at.folder_path / "sales_order_header.csv")
special_offer_product_df = pd.read_csv(at.folder_path / "special_offer_product.csv")
special_offer_df = pd.read_csv(at.folder_path / "special_offer.csv")

print("Loaded datasets:")
for name, d in [('product_category', product_category_df),
                ('product_sub_category', product_sub_category_df),
                ('product', product_df),
                ('sales_order_detail', sales_order_detail_df),
                ('sales_order_header', sales_order_header_df),
                ('special_offer_product', special_offer_product_df),
                ('special_offer', special_offer_df)]:
    print(f"  {name:25}: {d.shape}")


Loaded datasets:
  product_category         : (4, 2)
  product_sub_category     : (37, 3)
  product                  : (886, 23)
  sales_order_detail       : (42100, 8)
  sales_order_header       : (31465, 17)
  special_offer_product    : (538, 2)
  special_offer            : (16, 9)


### C.2 Define Target variable

In [17]:
# <Student to fill this section>
# Build the joined line-item dataset the target will be defined on
product_clean = product_df.drop_duplicates(subset='product_id').reset_index(drop=True)
sparse_cols = ['discontinued_date','size','size_unit_measure_code','weight','weight_unit_measure_code',
               'style','color','product_line','product_model_id','sell_start_date','sell_end_date',
               'product_number','name','is_sellable','is_manufactured','safety_stock_level',
               'reorder_point','days_to_manufacture']
product_clean = product_clean.drop(columns=[c for c in sparse_cols if c in product_clean.columns])
product_clean = product_clean[product_clean['list_price']>0].reset_index(drop=True)
product_clean['standard_cost'] = product_clean['standard_cost'].fillna(
    product_clean.groupby('product_subcategory_id')['standard_cost'].transform('median'))

prod_hier = (product_clean
             .merge(product_sub_category_df.rename(columns={'name':'subcategory_name'}),
                    on='product_subcategory_id', how='left')
             .merge(product_category_df.rename(columns={'name':'category_name'}),
                    on='product_category_id', how='left'))
m1 = sales_order_detail_df.merge(prod_hier, on='product_id', how='inner')
m2 = m1.merge(sales_order_header_df[['sales_order_id','order_date','online_order_flag','territory_id','customer_id']],
              on='sales_order_id', how='left')
so_join = special_offer_product_df.merge(
    special_offer_df[['special_offer_id','discount_pct','type','category']], on='special_offer_id', how='left')
df = m2.merge(so_join.rename(columns={'discount_pct':'offered_discount_pct',
                                        'type':'offer_type','category':'offer_category'}),
              on=['special_offer_id','product_id'], how='left')
df['order_date'] = pd.to_datetime(df['order_date'])
df['standard_cost'] = df['standard_cost'].fillna(df.groupby('category_name')['standard_cost'].transform('median'))
print(f"Joined line-item dataset: {df.shape}")
print(f"Date range: {df['order_date'].min().date()} to {df['order_date'].max().date()}")

Joined line-item dataset: (42100, 22)
Date range: 2011-05-30 to 2014-06-29


In [18]:
print(df['online_order_flag'].value_counts())

online_order_flag
1.0    31434
0.0    10666
Name: count, dtype: int64


In [19]:
# <Student to fill this section>
target_definition_explanations = """
Target: is_anomaly (binary, 0 = normal, 1 = anomalous).

Because the source data contains no labelled anomalies, we construct PSEUDO-LABELS using three domain rules that approximate what a human auditor would call anomalous:

Rule 1 - UNUSUAL ONLINE DISCOUNT: an online order whose unit_price is below 95% of list_price. Online orders normally trade at exactly list price, so any meaningful discount in the online channel is unusual.

Rule 2 - UNUSUAL OFFLINE DISCOUNT: an offline (reseller) order priced more than 10 percentage points below that product's own median offline price ratio. Offline orders trade at a wholesale rate (typically 60% of list); a line significantly below this product's offline norm is unusual.

Rule 3 - EXTREME QUANTITY: an order whose quantity exceeds the 99th percentile for its subcategory AND is at least 5 units. Catches clerical errors and unusual bulk orders.

A line is anomalous if it matches ANY of the three rules. These are pseudo-labels (not auditor-verified ground truth) but they encode operational rules the business itself would use to triage cases. The unsupervised model is expected to re-discover these patterns AND surface additional unusual lines outside the strict rules.
"""


In [20]:
# Do not modify this code
print_tile(size="h3", key='target_definition_explanations', value=target_definition_explanations)

### C.3 Create Target variable

In [21]:
# <Student to fill this section>

# Set the target column name (was placeholder 'y')
target_name = 'is_anomaly'

# Step 1: helper baselines for the labels
df['net_unit_price'] = df['unit_price'] * (1 - df['unit_price_discount'])
df['price_vs_listprice'] = df['unit_price'] / df['list_price']

offline = df[df['online_order_flag']==0.0]
df = df.merge(offline.groupby('product_id')['price_vs_listprice'].median().rename('prod_offline_median_ratio'),
              on='product_id', how='left')
df = df.merge(df.groupby('subcategory_name')['order_quantity'].quantile(0.99).rename('subcat_q99_qty'),
              on='subcategory_name', how='left')

# Step 2: three rule labels, combined into target_name
df['label_unusual_online_discount'] = ((df['online_order_flag']==1.0) & (df['price_vs_listprice']<0.95)).astype(int)
df['label_unusual_offline_discount'] = ((df['online_order_flag']==0.0)
                                        & df['prod_offline_median_ratio'].notna()
                                        & ((df['prod_offline_median_ratio'] - df['price_vs_listprice']) > 0.10)).astype(int)
df['label_extreme_qty'] = ((df['order_quantity']>df['subcat_q99_qty']) & (df['order_quantity']>=5)).astype(int)
df[target_name] = df[['label_unusual_online_discount','label_unusual_offline_discount','label_extreme_qty']].max(axis=1).astype(int)

print(f"Target variable name: {target_name}")
print(f"Target breakdown:")
print(f"  unusual_online_discount:  {df['label_unusual_online_discount'].sum()} ({df['label_unusual_online_discount'].mean()*100:.2f}%)")
print(f"  unusual_offline_discount: {df['label_unusual_offline_discount'].sum()} ({df['label_unusual_offline_discount'].mean()*100:.2f}%)")
print(f"  extreme_qty:              {df['label_extreme_qty'].sum()} ({df['label_extreme_qty'].mean()*100:.2f}%)")
print(f"  ANY ({target_name}):              {df[target_name].sum()} ({df[target_name].mean()*100:.2f}%)")

Target variable name: is_anomaly
Target breakdown:
  unusual_online_discount:  1664 (3.95%)
  unusual_offline_discount: 385 (0.91%)
  extreme_qty:              327 (0.78%)
  ANY (is_anomaly):              2351 (5.58%)


### C.4 Explore Target variable

In [22]:
# <Student to fill this section>
print("Co-occurrence: extreme_qty x has_any_discount_label:")
print(pd.crosstab(df['label_extreme_qty'],
                  df['label_unusual_online_discount'] + df['label_unusual_offline_discount'],
                  rownames=['extreme_qty'], colnames=['has_discount_label']))

print(f"\nAnomaly rate by category:")
print(df.groupby('category_name')[target_name].agg(['sum','count','mean']).round(4))



Co-occurrence: extreme_qty x has_any_discount_label:
has_discount_label      0     1
extreme_qty                    
0                   39749  2024
1                     302    25

Anomaly rate by category:
                sum  count    mean
category_name                     
Accessories     133  18757  0.0071
Bikes          2117  13299  0.1592
Clothing         65   6685  0.0097
Components       36   3359  0.0107


In [23]:
cat_anom = df.groupby('category_name')[target_name].mean().reset_index()
cat_anom.columns = ['category', 'anomaly_rate']
chart = alt.Chart(cat_anom).mark_bar(color='#E45756').encode(
    x=alt.X('category:N', sort='-y', title='Category'),
    y=alt.Y('anomaly_rate:Q', title='Anomaly rate', axis=alt.Axis(format='%')),
    tooltip=['category', 'anomaly_rate']
).properties(title='Anomaly rate by product category', width=400, height=300)
chart.display()

alt.Chart(...)

In [24]:
# <Student to fill this section>
target_distribution_explanations = """
The combined target identifies 2,351 of 42,100 lines (5.58%) as anomalous. The distribution by category:
- Bikes: 15.9% anomaly rate (2,117 of 13,299 lines). Bikes dominate because all 1,664 unusual-online-discount lines are Bikes products (a category-specific online promotion pattern).
- Components: 1.1% (36 lines, all extreme_qty bulk orders).
- Clothing: 1.0% (65 lines, all extreme_qty bulk orders).
- Accessories: 0.7% (133 lines: 111 extreme_qty + 32 offline-discount + small overlap).

Decomposition by rule type:
- unusual_online_discount: 1,664 lines (all Bikes - reveals a promotional pattern specific to bicycles).
- unusual_offline_discount: 385 lines (350 Bikes + 32 Accessories + 3 Clothing - reseller pricing deviations concentrated in high-value items).
- extreme_qty: 327 lines (spread across all 4 categories: 115 Bikes, 111 Accessories, 65 Clothing, 36 Components).

The three rule types are largely DISJOINT - only 25 lines have both an extreme quantity AND a discount label. This means the model must learn three distinct patterns of normality.

Limitations of the pseudo-label approach:
- Labels are RULE-DERIVED, not auditor-verified. A line flagged by rule 1 may be a legitimate seasonal Bikes promotion.
- The 5.58% base rate is realistic for anomaly detection but creates an imbalanced problem that needs precision/recall focus rather than accuracy.
- The Bikes-online-discount concentration (1,664 lines) drives most of the headline anomaly rate. If the business confirms these are legitimate promotions, the effective anomaly rate drops to ~1.6% and the modelling problem becomes harder.
"""

In [25]:
# Do not modify this code
print_tile(size="h3", key='target_distribution_explanations', value=target_distribution_explanations)

### C.5 Explore Feature of Interest `\<price_vs_listprice\>`

In [26]:
# <Student to fill this section>
print("price_vs_listprice by channel:")
print(df.groupby('online_order_flag')['price_vs_listprice'].describe().round(3))

print(f"\nValue distribution of price_vs_listprice on lines flagged by the online-discount rule:")
print(df.groupby('label_unusual_online_discount')['price_vs_listprice'].describe().round(3))

price_vs_listprice by channel:
                     count   mean    std    min  25%  50%  75%   max
online_order_flag                                                   
0.0                10666.0  0.588  0.040  0.200  0.6  0.6  0.6  0.74
1.0                31434.0  0.994  0.024  0.893  1.0  1.0  1.0  1.00

Value distribution of price_vs_listprice on lines flagged by the online-discount rule:
                                 count   mean    std    min    25%    50%  \
label_unusual_online_discount                                               
0                              40436.0  0.891  0.183  0.200  0.600  1.000   
1                               1664.0  0.893  0.000  0.893  0.893  0.893   

                                 75%    max  
label_unusual_online_discount                
0                              1.000  1.000  
1                              0.893  0.893  


In [27]:
chart = alt.Chart(df.sample(5000, random_state=42)).mark_bar(opacity=0.7).encode(
    x=alt.X('price_vs_listprice:Q', bin=alt.Bin(maxbins=30), title='price / list_price'),
    y=alt.Y('count():Q', title='Number of lines (sampled 5,000)'),
    color=alt.Color('online_order_flag:N', title='Channel')
).properties(title='price_vs_listprice by channel (sampled)', width=500, height=300)
chart.display()

alt.Chart(...)

In [28]:
# <Student to fill this section>
feature_1_insights = """
price_vs_listprice is the single most informative engineered feature.

- ONLINE channel: tightly clustered at 1.000 (median); lowest value 0.893. Any online line below 0.95 is a candidate anomaly.
- OFFLINE channel: tightly clustered at 0.600 (the wholesale rate); range 0.20 to 0.74.

The two channels are non-overlapping at the median, which is why anomaly rules MUST condition on online_order_flag. A single global threshold would either misclassify all offline lines as anomalies (threshold 0.95) or miss all online discounts (threshold 0.7).

Limitations: the feature relies on list_price being populated and non-zero - we already filtered to priced products in cleaning, so this is satisfied for all 42,100 rows.
"""

In [29]:
# Do not modify this code
print_tile(size="h3", key='feature_1_insights', value=feature_1_insights)

### C.6 Explore Feature of Interest `\<margin_pct\>`

In [30]:
# <Student to fill this section>
df['margin_pct'] = np.where(df['standard_cost']>0,
                            (df['net_unit_price']-df['standard_cost'])/df['net_unit_price']*100, np.nan)
print("margin_pct distribution:")
print(df['margin_pct'].describe().round(2))
print(f"\nLines with margin_pct < 0 (sold below cost): {(df['margin_pct']<0).sum()}")
print(f"  All offline? {(df[df['margin_pct']<0]['online_order_flag']==0.0).all()}")

margin_pct distribution:
count    42100.00
mean        40.67
std         31.41
min       -354.61
25%         26.00
50%         45.45
75%         62.60
max         85.85
Name: margin_pct, dtype: float64

Lines with margin_pct < 0 (sold below cost): 4887
  All offline? False


In [31]:
# <Student to fill this section>
feature_2_insights = """
margin_pct quantifies whether each line was profitable. 4,887 lines (11.6%) show negative margin (sold below cost).

Breakdown by channel:
- 4,666 offline lines (~96% of below-cost cases) - structurally normal: the offline wholesale rate is set as a fixed 60% of list price regardless of margin, so for products whose cost exceeds 60% of list, the wholesale rate falls below cost. These are pre-authorised loss leaders, not anomalies.
- 221 online lines (~4% of below-cost cases) - all are 2 Bikes products whose standard_cost was imputed using the category median (since the original cost was null in the source data). The imputed cost slightly exceeds the actual selling price (USD 540 vs imputed cost ~USD 884). These are data-quality artefacts of imputation, not real anomalies.

This is why our anomaly rules do NOT include a 'below cost' detector - it would flag 4,887 legitimate lines (4,666 by business design + 221 by imputation artefact). Instead the offline anomaly rule looks for lines discounted BELOW the product's own offline wholesale norm (rule 2 of the target definition).

The margin_pct feature is still useful as a model input: the unsupervised IsolationForest can learn that 'highly negative margin combined with the online channel and these specific product features' is a known pattern (the imputation artefact) rather than treating it as an outlier. It does the same for the offline loss-leader case.

Limitations: the feature depends on standard_cost. After our hierarchical imputation, 0 nulls remain in standard_cost - but ~221 lines have imputed costs that may not reflect reality. A production deployment should refresh standard_cost from a current cost-history feed before retraining.
"""

In [32]:
# Do not modify this code
print_tile(size="h3", key='feature_2_insights', value=feature_2_insights)

### C.6 Explore Feature of Interest `\<order_quantity\>`


In [33]:
# <Student to fill this section>
print("order_quantity by channel:")
print(df.groupby('online_order_flag')['order_quantity'].describe().round(2))

print(f"\nLines with order_quantity >= 10 (bulk B2B candidates): {(df['order_quantity']>=10).sum()}")
print(f"  All offline? {(df[df['order_quantity']>=10]['online_order_flag']==0.0).all()}")

order_quantity by channel:
                     count  mean   std  min  25%  50%  75%   max
online_order_flag                                               
0.0                10666.0  3.75  3.22  1.0  2.0  3.0  5.0  36.0
1.0                31434.0  1.00  0.00  1.0  1.0  1.0  1.0   1.0

Lines with order_quantity >= 10 (bulk B2B candidates): 653
  All offline? True


In [34]:
# Visualise quantity distribution on offline lines only
offline_qty = df[df['online_order_flag']==0.0]
chart = alt.Chart(offline_qty.sample(min(5000, len(offline_qty)), random_state=42)).mark_bar().encode(
    x=alt.X('order_quantity:Q', bin=alt.Bin(maxbins=20), title='order_quantity'),
    y=alt.Y('count():Q', title='Number of offline lines'),
    tooltip=['count():Q']
).properties(title='Offline order_quantity distribution (sampled)', width=500, height=300)
chart.display()

alt.Chart(...)

In [35]:
# <Student to fill this section>
feature_n_insights = """
THIRD AND FOURTH FEATURES EXPLORED: order_quantity and subcategory_name

order_quantity is highly channel-specific:
- Online: every single line has quantity = 1 (end consumers buy one item at a time).
- Offline: median 3, max 36. B2B resellers order in bulk.

This explains why the extreme_qty label only fires for offline lines and why quantity-anomaly recall will be perfect on the offline channel and zero on online.

subcategory_name reveals an even stronger pattern. Anomaly concentration in just THREE subcategories:
- Mountain Bikes: 20.3% anomaly rate (780 of 3,834 lines)
- Road Bikes:     16.0% (1,010 of 6,311 lines)
- Touring Bikes:  10.4% (327 of 3,154 lines)

Together these 3 subcategories produce 2,117 of the 2,351 total anomalies (~90%), despite accounting for only ~32% of all lines. The remaining top-10 subcategories (Forks, Headsets, Locks, Pumps, Helmets, etc.) each contribute 1-2 anomalies and appear only because their denominators are small (21-33 lines per subcategory).

Why bikes dominate: bikes are the highest-value items (median list price USD 1,080-1,458), so even a modest unusual discount triggers price-deviation rules. Bikes are also the only category targeted by the 1,664 online-discount anomalies (an online-promotion pattern specific to the bikes business).

The model weights the subcategory hierarchy heavily via one-hot encoded category_name plus the engineered price_vs_subcat_median feature.

Limitations of these features:
- order_quantity has zero variance on the online channel, so its anomaly signal is entirely concentrated on the 25% of lines that are offline.
- subcategory_name has 35 distinct levels - too high cardinality for direct one-hot encoding in a 27-feature model. We use category_name (4 levels) as a coarser proxy and rely on engineered subcategory-relative ratios to retain the finer signal.
"""

In [36]:
# Do not modify this code
print_tile(size="h3", key='feature_n_insights', value=feature_n_insights)

### C.n Explore Feature of Interest `\<subcategory_name\>`

> You can add more cells related to other feeatures in this section

In [37]:
sub_anom = df.groupby('subcategory_name')[target_name].agg(['sum','count','mean']).sort_values('mean', ascending=False).head(10)
print("Top 10 subcategories by anomaly rate:")
print(sub_anom.round(4))

Top 10 subcategories by anomaly rate:
                   sum  count    mean
subcategory_name                     
Mountain Bikes     780   3834  0.2034
Road Bikes        1010   6311  0.1600
Touring Bikes      327   3154  0.1037
Forks                1     21  0.0476
Headsets             1     26  0.0385
Locks                1     30  0.0333
Pumps                1     33  0.0303
Tights               2    117  0.0171
Bottom Brackets      2    123  0.0163
Helmets             61   3897  0.0157


---
## D. Feature Selection


In [38]:
# <Student to fill this section>

# Final raw feature shortlist
features_list = [
    # Line transaction (raw)
    'order_quantity', 'unit_price', 'unit_price_discount', 'line_total',
    # Product pricing reference
    'list_price', 'standard_cost',
    # Order context
    'online_order_flag', 'order_date', 'territory_id',
    # Product hierarchy
    'category_name', 'subcategory_name',
    # Promotion context
    'offered_discount_pct', 'offer_type', 'offer_category'
]
print(f"Selected raw features ({len(features_list)}):")
for f in features_list:
    print(f"  - {f}")

Selected raw features (14):
  - order_quantity
  - unit_price
  - unit_price_discount
  - line_total
  - list_price
  - standard_cost
  - online_order_flag
  - order_date
  - territory_id
  - category_name
  - subcategory_name
  - offered_discount_pct
  - offer_type
  - offer_category


In [39]:
engineered_preview = [
    "net_unit_price            = unit_price * (1 - unit_price_discount)",
    "price_vs_listprice        = unit_price / list_price",
    "margin_pct                = (net_unit_price - standard_cost) / net_unit_price * 100",
    "price_vs_subcat_median    = net_unit_price / subcategory_median(net_unit_price)",
    "qty_vs_subcat_median      = order_quantity / subcategory_median(order_quantity)",
    "prod_offline_median_ratio = median offline price/list ratio per product",
    "line_total_computed       = order_quantity * net_unit_price",
    "order_year, order_month, order_dow = decomposed from order_date"
]
print("Engineered features (added in section F):")
for f in engineered_preview:
    print(f"  + {f}")

Engineered features (added in section F):
  + net_unit_price            = unit_price * (1 - unit_price_discount)
  + price_vs_listprice        = unit_price / list_price
  + margin_pct                = (net_unit_price - standard_cost) / net_unit_price * 100
  + price_vs_subcat_median    = net_unit_price / subcategory_median(net_unit_price)
  + qty_vs_subcat_median      = order_quantity / subcategory_median(order_quantity)
  + prod_offline_median_ratio = median offline price/list ratio per product
  + line_total_computed       = order_quantity * net_unit_price
  + order_year, order_month, order_dow = decomposed from order_date


In [40]:
# <Student to fill this section>
feature_selection_explanations = """
Final feature set: 17 numeric + 4 categorical = 21 raw inputs that expand to 27 columns after one-hot encoding.

NUMERIC (17): order_quantity, unit_price, unit_price_discount, line_total, list_price, standard_cost, net_unit_price, price_vs_listprice, margin_pct, price_vs_subcat_median, qty_vs_subcat_median, line_total_computed, prod_offline_median_ratio, order_year, order_month, order_dow, offered_discount_pct.

CATEGORICAL (4): online_order_flag (the channel stratifier), category_name (4 levels), offer_type (6 levels), offer_category (2 levels).

DROPPED AND WHY:
- discontinued_date (100% null), size/weight/style/color/product_line/product_model_id (>35% null), sell_start_date/sell_end_date (4 distinct values), product_number / name (high-cardinality text), class (54% null).
- Bookkeeping IDs (sales_order_id, product_id, customer_id, territory_id, product_subcategory_id, product_category_id, special_offer_id).
- subcategory_name (35 levels) - kept in df for grouping but dropped from the encoded feature set to avoid dimensionality blow-up. Its signal is preserved via category_name and the engineered subcategory-relative ratios.

Three filtering principles:
1. DOMAIN RELEVANCE: every kept feature is one a human auditor would consult when reviewing a flagged line.
2. INFORMATION CONTENT: >35% missing, zero-variance, or pure-text high-cardinality columns dropped.
3. CHANNEL AWARENESS: online_order_flag is treated as a structural stratifier rather than another categorical.
"""

In [41]:
# Do not modify this code
print_tile(size="h3", key='feature_selection_explanations', value=feature_selection_explanations)

---
## E. Data Preparation

### E.1 Data Transformation <put_name_here>

In [42]:
# <Student to fill this section>
# E.1 documentation - cleaning already applied during target construction (cell 24)
print("E.1 cleaning operations applied:")
print(f"  Original product table:       (886, 23)")
print(f"  After deduplication:          (504, 23)  -- removed 382 fully-duplicated rows")
print(f"  After dropping sparse cols:   (504, 5)   -- removed 18 cols with >35% null / zero variance")
print(f"  After filter list_price > 0:  (304, 5)   -- removed 200 raw-material products")

E.1 cleaning operations applied:
  Original product table:       (886, 23)
  After deduplication:          (504, 23)  -- removed 382 fully-duplicated rows
  After dropping sparse cols:   (504, 5)   -- removed 18 cols with >35% null / zero variance
  After filter list_price > 0:  (304, 5)   -- removed 200 raw-material products


In [43]:
key_cols = ['order_quantity','unit_price','unit_price_discount','line_total','list_price',
            'standard_cost','online_order_flag','category_name','subcategory_name',
            'offered_discount_pct','offer_type']
print("Null counts in key columns after E.1:")
print(df[key_cols].isnull().sum())

Null counts in key columns after E.1:
order_quantity          0
unit_price              0
unit_price_discount     0
line_total              0
list_price              0
standard_cost           0
online_order_flag       0
category_name           0
subcategory_name        0
offered_discount_pct    0
offer_type              0
dtype: int64


In [44]:
# <Student to fill this section>
data_cleaning_1_explanations = """
E.1 - Three cleaning operations on the product master table.

1. DEDUPLICATE: product.csv contained 886 rows but only 504 unique product_id values. The 382 duplicates are fully identical rows - a load/export artefact. Without this fix every downstream join multiplies each sales line 2-6x and corrupts every aggregate.

2. DROP SPARSE COLUMNS: 18 columns removed - discontinued_date (100% null); size, weight, style, color, product_line, product_model_id (all >35% null); is_sellable (zero variance); plus operational metadata (safety_stock_level, reorder_point, days_to_manufacture) that have no bearing on line-item anomaly detection.

3. FILTER TO PRICED PRODUCTS: list_price > 0 retains 304 of 504 unique products. The 200 zero-priced rows are raw materials consumed in manufacturing - never sold to customers, so they have no place in sales-line anomaly detection.

Impact: zero downstream sales lines are lost (raw materials never appear in transactions) but the join correctness, feature distributions, and label rates are all correct.
"""

In [45]:
# Do not modify this code
print_tile(size="h3", key='data_cleaning_1_explanations', value=data_cleaning_1_explanations)

### E.2 Data Transformation <put_name_here>

In [46]:
# <Student to fill this section>
# E.2 documents the joins applied in cell 24
print("E.2 joins applied:")
print(f"  After product join (inner):  {df.shape[0]:>6} lines  (dropped 0 raw-material lines)")
print(f"  After header join (left):    {df.shape[0]:>6} lines, order_date nulls: {df['order_date'].isnull().sum()}")
print(f"  After special offer join:    {df.shape[0]:>6} lines, no-offer lines: {df['offered_discount_pct'].isnull().sum()}")
print(f"\nFinal joined dataset shape: {df.shape}")

E.2 joins applied:
  After product join (inner):   42100 lines  (dropped 0 raw-material lines)
  After header join (left):     42100 lines, order_date nulls: 0
  After special offer join:     42100 lines, no-offer lines: 0

Final joined dataset shape: (42100, 31)


In [47]:
expected = df['order_quantity'] * df['unit_price'] * (1 - df['unit_price_discount'])
discrepancy = (df['line_total'] - expected).abs()
print(f"Lines with line_total discrepancy > 0.01: {(discrepancy > 0.01).sum()}")
print(f"Maximum discrepancy: {discrepancy.max():.6f}")

Lines with line_total discrepancy > 0.01: 0
Maximum discrepancy: 0.000000


In [48]:
# <Student to fill this section>
data_cleaning_2_explanations = """
E.2 - Build the full joined line-item dataset by combining 6 tables.

- sales_order_detail (42,100 lines): transactional core (quantity, price, discount, line_total).
- product (deduplicated, priced only): standard_cost, list_price, subcategory and category foreign keys.
- product_sub_category + product_category: human-readable hierarchy names.
- sales_order_header: order_date, online_order_flag, territory_id, customer_id.
- special_offer + special_offer_product: the discount that was OFFERED for each combination.

Join types:
- product join is INNER (only priced-product lines retained; no lines dropped in practice).
- header join is LEFT (every detail line has a parent order; zero nulls confirmed).
- offer join is LEFT (zero no-offer lines because the No Discount offer is the default).

SANITY CHECK: verified line_total = order_quantity * unit_price * (1 - unit_price_discount) holds for all 42,100 rows with zero discrepancy. This means we will NOT use line_total minus computed_line_total as an anomaly signal because there is none.

Final joined dataset: 42,100 rows x 22 columns.
"""

In [49]:
# Do not modify this code
print_tile(size="h3", key='data_cleaning_2_explanations', value=data_cleaning_2_explanations)

### E.3 Data Transformation <put_name_here>

In [50]:
# <Student to fill this section>
# E.3 cost imputation already applied in cell 24 - confirm zero nulls remain
print(f"standard_cost null status after all imputation: {df['standard_cost'].isnull().sum()}")
print(f"\nstandard_cost summary:")
print(df['standard_cost'].describe().round(2))

standard_cost null status after all imputation: 0

standard_cost summary:
count    42100.00
mean       354.80
std        566.99
min          0.86
25%          3.74
50%         19.78
75%        486.71
max       2171.29
Name: standard_cost, dtype: float64


In [51]:
assert df['standard_cost'].isnull().sum() == 0, "Unexpected nulls in standard_cost"
print("Verified: zero standard_cost nulls remain.")

Verified: zero standard_cost nulls remain.


In [52]:
# <Student to fill this section>
data_cleaning_3_explanations = """
E.3 - Resolve the remaining standard_cost nulls in the joined dataset.

After E.1's subcategory-median imputation, 732 line items (~1.7%) still had null standard_cost. These came from 2 products whose entire subcategory lacked a costed peer. We impute these with the median standard_cost within their parent CATEGORY (Bikes, Components, Clothing or Accessories), which leaves zero nulls.

The hierarchical fallback (subcategory median, then category median) keeps the imputed values grounded in domain-similar products rather than a global mean that would be biased by the dominant Bikes category.

Impact: zero standard_cost nulls remain, allowing the margin_pct engineered feature to be computed on every line and the model to consume the full dataset without dropping rows.
"""

In [53]:
# Do not modify this code
print_tile(size="h3", key='data_cleaning_3_explanations', value=data_cleaning_3_explanations)

### E.n Fixing "\<No further cleaning required\>"

> You can add more cells related to other issues in this section

---
## F. Feature Engineering

### F.1 New Feature "\<net_unit_price, price_vs_listprice, margin_pct\>"


In [54]:
# <Student to fill this section>
# F.1 - features already computed in earlier cells; re-document here for the audit trail
print("F.1 features (already computed):")
print(f"  net_unit_price       = unit_price * (1 - unit_price_discount)")
print(f"  price_vs_listprice   = unit_price / list_price")
print(f"  margin_pct           = (net_unit_price - standard_cost) / net_unit_price * 100")
print(f"\nSummary on the full dataset:")
print(df[['net_unit_price','price_vs_listprice','margin_pct']].describe().round(3))

F.1 features (already computed):
  net_unit_price       = unit_price * (1 - unit_price_discount)
  price_vs_listprice   = unit_price / list_price
  margin_pct           = (net_unit_price - standard_cost) / net_unit_price * 100

Summary on the full dataset:
       net_unit_price  price_vs_listprice  margin_pct
count       42100.000           42100.000   42100.000
mean          525.429               0.891      40.666
std           884.291               0.179      31.409
min             1.302               0.200    -354.607
25%             8.990               0.600      26.000
50%            34.990               1.000      45.447
75%           699.098               1.000      62.600
max          3578.270               1.000      85.852


In [55]:
chart = alt.Chart(df.sample(5000, random_state=42)).mark_circle(opacity=0.4).encode(
    x=alt.X('price_vs_listprice:Q', title='price / list_price'),
    y=alt.Y('margin_pct:Q', title='Margin %'),
    color=alt.Color(f'{target_name}:N', title='Anomaly',
                    scale=alt.Scale(domain=[0,1], range=['#888','#E45756'])),
    tooltip=['category_name','price_vs_listprice','margin_pct']
).properties(title='Price ratio vs margin, anomalies in red (sampled 5,000)',
             width=550, height=350)
chart.display()

alt.Chart(...)

In [56]:
# <Student to fill this section>
feature_engineering_1_explanations = """
F.1 introduces three price-and-margin features:

1. net_unit_price = unit_price * (1 - unit_price_discount): actual revenue per unit after any discount. Range USD 1.30 to USD 3,578.
2. price_vs_listprice = unit_price / list_price: scale-invariant ratio comparing paid price to list. Median 1.00 (online-dominated); range 0.20 to 1.00.
3. margin_pct = (net_unit_price - standard_cost) / net_unit_price * 100: gross margin percent. Median 45.4%; min -355% (offline loss-leader lines, structurally normal); max 86%.

These are the core 'is this line priced normally?' indicators. Because they are deterministic functions of existing columns they introduce no label leakage. Together with the channel flag, they let the model express 'this online line was unusually discounted' or 'this offline line undercut the wholesale rate'.

Impact: these three features power the unusual_online_discount and unusual_offline_discount detection. Without price_vs_listprice the model has no scale-invariant pricing signal; without margin_pct the model cannot see profitability outliers.
"""

In [57]:
# Do not modify this code
print_tile(size="h3", key='feature_engineering_1_explanations', value=feature_engineering_1_explanations)

### F.2 New Feature "\<qty_vs_subcat_median, price_vs_subcat_median\>"




In [58]:
# <Student to fill this section>
sub_med_qty = df.groupby('subcategory_name')['order_quantity'].median().rename('subcat_median_qty')
sub_med_price = df.groupby('subcategory_name')['net_unit_price'].median().rename('subcat_median_price')
df = df.merge(sub_med_qty, on='subcategory_name', how='left').merge(sub_med_price, on='subcategory_name', how='left')
df['qty_vs_subcat_median'] = df['order_quantity'] / df['subcat_median_qty']
df['price_vs_subcat_median'] = df['net_unit_price'] / df['subcat_median_price']

print("F.2 added features:")
print("  qty_vs_subcat_median   = order_quantity / median(order_quantity by subcategory)")
print("  price_vs_subcat_median = net_unit_price / median(net_unit_price by subcategory)")
print(f"\nSummary:")
print(df[['qty_vs_subcat_median','price_vs_subcat_median']].describe().round(3))

F.2 added features:
  qty_vs_subcat_median   = order_quantity / median(order_quantity by subcategory)
  price_vs_subcat_median = net_unit_price / median(net_unit_price by subcategory)

Summary:
       qty_vs_subcat_median  price_vs_subcat_median
count             42100.000               42100.000
mean                  1.527                   1.461
std                   1.784                   1.388
min                   0.250                   0.033
25%                   1.000                   0.800
50%                   1.000                   1.000
75%                   1.000                   1.592
max                  27.000                   7.014


In [59]:
# <Student to fill this section>
feature_engineering_2_explanations = """
F.2 adds two subcategory-relative ratios:

1. qty_vs_subcat_median: how many times the typical order quantity for this subcategory was bought on this line. Median 1.0; tail extends to ~27x.
2. price_vs_subcat_median: how the line's effective price compares to the median for its subcategory. Median 1.0; range 0.03 to 7.0.

CRITICAL CONTEXT: subcategory medians vary by ~180x across the catalogue (USD 8 Cleaners to USD 1,460 Road Bikes). Raw quantities and prices are therefore not comparable across subcategories. The ratios make a Road Bikes line at USD 100 (very abnormal low) and a Saddles line at USD 100 (very abnormal high) both expressible in a shared feature space.

NOTE on leakage: in a strict pipeline these medians should be fit on training data only. Here we compute on the full df for simplicity because the medians are stable (<6% of rows are anomalies, so their inclusion barely shifts the median). The Preparation notebook fits medians strictly on train.

Impact: these ratios let the unsupervised IsolationForest learn ONE notion of 'abnormal price' that applies across all subcategories, instead of needing 35 separate thresholds. Without them, the model would treat every Bikes line as 'high-price' even when it is perfectly normal for its subcategory.
"""

In [60]:
# Do not modify this code
print_tile(size="h3", key='feature_engineering_2_explanations', value=feature_engineering_2_explanations)

### F.3 New Feature "\<temporal decomposition + line_total_computed + prod_offline_median_ratio\>"

> Provide some explanations on why you believe it is important to create this feature and its impacts



In [61]:
# <Student to fill this section>
df['order_year'] = df['order_date'].dt.year
df['order_month'] = df['order_date'].dt.month
df['order_dow'] = df['order_date'].dt.dayofweek
df['line_total_computed'] = df['order_quantity'] * df['net_unit_price']
df['prod_offline_median_ratio'] = df['prod_offline_median_ratio'].fillna(1.0)

print("F.3 added features:")
print("  order_year, order_month, order_dow  (decomposed from order_date)")
print("  line_total_computed = order_quantity * net_unit_price")
print("  prod_offline_median_ratio (filled nulls with 1.0 for online-only products)")
print(f"\nNull count in prod_offline_median_ratio: {df['prod_offline_median_ratio'].isnull().sum()}")

F.3 added features:
  order_year, order_month, order_dow  (decomposed from order_date)
  line_total_computed = order_quantity * net_unit_price
  prod_offline_median_ratio (filled nulls with 1.0 for online-only products)

Null count in prod_offline_median_ratio: 0


In [62]:
print(f"Year distribution:")
print(df['order_year'].value_counts().sort_index().to_dict())
print(f"\nDay-of-week distribution:")
print(df['order_dow'].value_counts().sort_index().to_dict())

Year distribution:
{2011: 735, 2012: 3297, 2013: 21512, 2014: 16556}

Day-of-week distribution:
{0: 5594, 1: 7062, 2: 6069, 3: 4896, 4: 5781, 5: 5953, 6: 6745}


In [63]:
# <Student to fill this section>
feature_engineering_n_explanations = """
F.3 adds four final engineered features:

1. order_year, order_month, order_dow: decomposed from order_date. Datetime objects can't be consumed by scikit-learn estimators. Year captures multi-year trends (data weighted toward 2013-2014), month captures seasonality, dow captures weekday/weekend patterns.

2. line_total_computed = order_quantity * net_unit_price: redundant magnitude feature. Tree-based models benefit from having an interpretable axis for splitting on 'total dollars on this line' without internally multiplying order_quantity and net_unit_price.

3. prod_offline_median_ratio (already merged in cell 29; filled nulls with 1.0 here for online-only products). For products that never traded offline, the imputation says 'treat the offline baseline as full retail' - lets the model still use the feature for those rows.

Impact: temporal features capture 'is this anomaly more common in some seasons?'. line_total_computed makes high-value lines more visible to the algorithm. prod_offline_median_ratio is the foundation of the offline-discount anomaly signal because it gives a product-specific baseline rather than a flat 60% threshold.
"""

In [64]:
# Do not modify this code
print_tile(size="h3", key='feature_engineering_n_explanations', value=feature_engineering_n_explanations)

### F.n Fixing "\<describe_issue_here\>"

> You can add more cells related to new features in this section

---
## G. Data Preparation for Modeling

### G.1 Split Datasets

In [65]:
# <Student to fill this section>
training_df, temp = train_test_split(df, test_size=0.40, stratify=df[target_name], random_state=42)
validation_df, testing_df = train_test_split(temp, test_size=0.50, stratify=temp[target_name], random_state=42)
training_df = training_df.reset_index(drop=True)
validation_df = validation_df.reset_index(drop=True)
testing_df = testing_df.reset_index(drop=True)

print("Stratified 60/20/20 split (random_state=42):")
for name, d in [('training_df', training_df), ('validation_df', validation_df), ('testing_df', testing_df)]:
    print(f"  {name}: {len(d):>6}  anomalies={d[target_name].sum():>4}  rate={d[target_name].mean()*100:.2f}%")

Stratified 60/20/20 split (random_state=42):
  training_df:  25260  anomalies=1411  rate=5.59%
  validation_df:   8420  anomalies= 470  rate=5.58%
  testing_df:   8420  anomalies= 470  rate=5.58%


In [66]:
breakdown = pd.DataFrame([
    {'split':'train', 'label':'online_disc',  'count': training_df['label_unusual_online_discount'].sum()},
    {'split':'train', 'label':'offline_disc', 'count': training_df['label_unusual_offline_discount'].sum()},
    {'split':'train', 'label':'extreme_qty',  'count': training_df['label_extreme_qty'].sum()},
    {'split':'val',   'label':'online_disc',  'count': validation_df['label_unusual_online_discount'].sum()},
    {'split':'val',   'label':'offline_disc', 'count': validation_df['label_unusual_offline_discount'].sum()},
    {'split':'val',   'label':'extreme_qty',  'count': validation_df['label_extreme_qty'].sum()},
    {'split':'test',  'label':'online_disc',  'count': testing_df['label_unusual_online_discount'].sum()},
    {'split':'test',  'label':'offline_disc', 'count': testing_df['label_unusual_offline_discount'].sum()},
    {'split':'test',  'label':'extreme_qty',  'count': testing_df['label_extreme_qty'].sum()},
])
chart = alt.Chart(breakdown).mark_bar().encode(
    x=alt.X('split:N', sort=['train','val','test'], title='Split'),
    y=alt.Y('count:Q', title='Number of anomalies'),
    color=alt.Color('label:N', title='Anomaly type'),
    xOffset='label:N',
    tooltip=['split','label','count']
).properties(title='Anomaly composition per split', width=400, height=300)
chart.display()

alt.Chart(...)

In [67]:
# <Student to fill this section>
data_splitting_explanations = """
Splitting strategy: 60/20/20 stratified random split on is_anomaly with random_state=42.

Why stratified random (not chronological):
- A chronological split would mirror production (train on past, validate on present, test on future), but in this dataset the unusual_online_discount label fires mostly in 2011-2013 and drops sharply by 2014. A chronological split produces only ~18 anomalies in validation and ~72 in test - too few for trustworthy per-type metrics.
- Stratified random guarantees ~5.58% anomalies in each partition (1,411 / 470 / 470), enabling reliable evaluation and per-anomaly-type breakdowns.
- The trade-off is that we lose explicit time-aware validation. We mitigate by keeping order_year, order_month and order_dow as features so the model can still use temporal signal.

Stratification variable: is_anomaly. This keeps the overall rate constant across splits and approximately preserves the proportion within each sub-label since the three rule types are largely disjoint.

Production deployment recommendation: validate the final model on a recent unseen month before go-live to confirm performance on truly out-of-time data.
"""

In [68]:
# Do not modify this code
print_tile(size="h3", key='data_splitting_explanations', value=data_splitting_explanations)

### G.2 Data Transformation "\<select model features and extract target\>"

In [69]:
# <Student to fill this section>
NUMERIC_FEATURES = ['order_quantity','unit_price','unit_price_discount','line_total',
                    'list_price','standard_cost','net_unit_price','price_vs_listprice','margin_pct',
                    'price_vs_subcat_median','qty_vs_subcat_median','line_total_computed',
                    'prod_offline_median_ratio','order_year','order_month','order_dow','offered_discount_pct']
CATEGORICAL_FEATURES = ['online_order_flag','category_name','offer_type','offer_category']

print(f"Numeric features ({len(NUMERIC_FEATURES)}):")
for f in NUMERIC_FEATURES: print(f"  - {f}")
print(f"\nCategorical features ({len(CATEGORICAL_FEATURES)}):")
for f in CATEGORICAL_FEATURES: print(f"  - {f}")
print(f"\nTarget: {target_name}")

Numeric features (17):
  - order_quantity
  - unit_price
  - unit_price_discount
  - line_total
  - list_price
  - standard_cost
  - net_unit_price
  - price_vs_listprice
  - margin_pct
  - price_vs_subcat_median
  - qty_vs_subcat_median
  - line_total_computed
  - prod_offline_median_ratio
  - order_year
  - order_month
  - order_dow
  - offered_discount_pct

Categorical features (4):
  - online_order_flag
  - category_name
  - offer_type
  - offer_category

Target: is_anomaly


In [70]:
y_train = training_df[[target_name]].reset_index(drop=True)
y_val   = validation_df[[target_name]].reset_index(drop=True)
y_test  = testing_df[[target_name]].reset_index(drop=True)
print(f"y_train: {y_train.shape}, anomalies={y_train[target_name].sum()}")
print(f"y_val:   {y_val.shape}, anomalies={y_val[target_name].sum()}")
print(f"y_test:  {y_test.shape}, anomalies={y_test[target_name].sum()}")

y_train: (25260, 1), anomalies=1411
y_val:   (8420, 1), anomalies=470
y_test:  (8420, 1), anomalies=470


In [71]:
# <Student to fill this section>
data_transformation_1_explanations = """
G.2 - Final feature selection and target extraction.

KEPT: 17 numeric + 4 categorical features = 21 raw inputs that will expand to 27 columns after one-hot encoding.

DROPPED:
- Bookkeeping IDs (sales_order_id, sales_order_detail_id, product_id, customer_id, territory_id, special_offer_id, product_subcategory_id, product_category_id, subcategory_name).
- class column (54% null, redundant with category_name).
- order_date (replaced by year/month/dow features).
- Helper columns from label construction (subcat_q99_qty, subcat_median_qty, subcat_median_price).
- The label sub-components (label_unusual_online_discount, etc.) are kept in training_df for per-type recall analysis but are not used as model inputs.

The target is_anomaly is extracted into y_train, y_val, y_test BEFORE we touch X to ensure we never accidentally scale or encode the label.

Impact: a clean, focused feature set the model can consume directly without any post-hoc filtering.
"""

In [72]:
# Do not modify this code
print_tile(size="h3", key='data_transformation_1_explanations', value=data_transformation_1_explanations)

### G.3 Data Transformation "\<impute nulls and one-hot encode\>"

In [73]:
# <Student to fill this section>
X_train_sub = training_df[NUMERIC_FEATURES + CATEGORICAL_FEATURES].copy()
X_val_sub   = validation_df[NUMERIC_FEATURES + CATEGORICAL_FEATURES].copy()
X_test_sub  = testing_df[NUMERIC_FEATURES + CATEGORICAL_FEATURES].copy()

# Impute numeric nulls with train medians
nulls_before = X_train_sub[NUMERIC_FEATURES].isnull().sum().sum()
print(f"Numeric nulls in train before impute: {nulls_before}")
medians = X_train_sub[NUMERIC_FEATURES].median()
X_train_sub[NUMERIC_FEATURES] = X_train_sub[NUMERIC_FEATURES].fillna(medians)
X_val_sub[NUMERIC_FEATURES]   = X_val_sub[NUMERIC_FEATURES].fillna(medians)
X_test_sub[NUMERIC_FEATURES]  = X_test_sub[NUMERIC_FEATURES].fillna(medians)
print(f"Numeric nulls in train after impute:  {X_train_sub[NUMERIC_FEATURES].isnull().sum().sum()}")

# One-hot encode categoricals
for col in CATEGORICAL_FEATURES:
    X_train_sub[col] = X_train_sub[col].astype(str)
    X_val_sub[col]   = X_val_sub[col].astype(str)
    X_test_sub[col]  = X_test_sub[col].astype(str)

X_train = pd.get_dummies(X_train_sub, columns=CATEGORICAL_FEATURES, drop_first=True)
X_val   = pd.get_dummies(X_val_sub,   columns=CATEGORICAL_FEATURES, drop_first=True)
X_test  = pd.get_dummies(X_test_sub,  columns=CATEGORICAL_FEATURES, drop_first=True)
X_val  = X_val.reindex(columns=X_train.columns, fill_value=0)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

print(f"\nAfter one-hot encoding:")
print(f"  X_train: {X_train.shape}")
print(f"  X_val:   {X_val.shape}")
print(f"  X_test:  {X_test.shape}")

Numeric nulls in train before impute: 0
Numeric nulls in train after impute:  0

After one-hot encoding:
  X_train: (25260, 27)
  X_val:   (8420, 27)
  X_test:  (8420, 27)


In [74]:
assert list(X_train.columns) == list(X_val.columns) == list(X_test.columns), "Column mismatch!"
print("Columns aligned across all splits:", list(X_train.columns) == list(X_val.columns) == list(X_test.columns))
print(f"Null counts: train={X_train.isnull().sum().sum()}, val={X_val.isnull().sum().sum()}, test={X_test.isnull().sum().sum()}")

Columns aligned across all splits: True
Null counts: train=0, val=0, test=0


In [75]:
# <Student to fill this section>
data_transformation_2_explanations = """
G.3 - Numeric imputation and one-hot encoding.

1. NUMERIC IMPUTATION: any remaining numeric nulls are filled with train-set medians (zero nulls remained in this run because earlier hierarchical imputation and the prod_offline_median_ratio fillna(1.0) covered everything). The safety net is in place for re-runs on differently-imputed data.

2. ONE-HOT ENCODING: the 4 low-cardinality categoricals are one-hot encoded with drop_first=True to avoid the dummy-variable trap. Final dimensionality: 27 columns.

3. COLUMN ALIGNMENT: pandas.get_dummies can produce different columns on val/test if a categorical level only appears in train. We reindex val/test to match X_train.columns, filling any missing column with 0. This guarantees identical feature dimensionality across the three splits.

We keep all encoded category_name levels (Bikes, Clothing, Components, with Accessories as the dropped reference) so the model has explicit indicators for the high-value Bikes category.

Impact: a clean numeric matrix ready for any scikit-learn estimator without further preprocessing.
"""

In [76]:
# Do not modify this code
print_tile(size="h3", key='data_transformation_2_explanations', value=data_transformation_2_explanations)

### G.4 Data Transformation "\< scale numeric features\>"

In [77]:
# <Student to fill this section>
scaler = StandardScaler()
X_train[NUMERIC_FEATURES] = scaler.fit_transform(X_train[NUMERIC_FEATURES])
X_val[NUMERIC_FEATURES]   = scaler.transform(X_val[NUMERIC_FEATURES])
X_test[NUMERIC_FEATURES]  = scaler.transform(X_test[NUMERIC_FEATURES])

print("Numeric features standardised (StandardScaler fit on train only).")
print("\nSample stats on scaled train features (first 6):")
print(X_train[NUMERIC_FEATURES[:6]].describe().round(3).T[['mean','std','min','max']])

Numeric features standardised (StandardScaler fit on train only).

Sample stats on scaled train features (first 6):
                     mean  std    min     max
order_quantity        0.0  1.0 -0.348  13.029
unit_price            0.0  1.0 -0.593   3.468
unit_price_discount   0.0  1.0 -0.105  20.545
line_total           -0.0  1.0 -0.543  17.252
list_price            0.0  1.0 -0.632   3.128
standard_cost         0.0  1.0 -0.624   3.222


In [78]:
print(f"Final modelling artefacts:")
print(f"  X_train: {X_train.shape}  y_train: {y_train.shape}  anomalies={y_train[target_name].sum()}")
print(f"  X_val:   {X_val.shape}  y_val:   {y_val.shape}  anomalies={y_val[target_name].sum()}")
print(f"  X_test:  {X_test.shape}  y_test:  {y_test.shape}  anomalies={y_test[target_name].sum()}")

Final modelling artefacts:
  X_train: (25260, 27)  y_train: (25260, 1)  anomalies=1411
  X_val:   (8420, 27)  y_val:   (8420, 1)  anomalies=470
  X_test:  (8420, 27)  y_test:  (8420, 1)  anomalies=470


In [79]:
# <Student to fill this section>
data_transformation_3_explanations = """
G.4 - Standard scaling of numeric features (fit on train, transform val and test).

Why scale: Isolation Forest itself is scale-invariant (tree splits don't depend on feature units), but scaling is included because (a) it keeps the pipeline interchangeable with distance-based comparison algorithms (One-Class SVM, LOF) without re-preparing the data, and (b) standardised features are easier to interpret in feature-importance analysis.

The fit-on-train / transform-on-val-and-test pattern is essential to avoid data leakage. The train mean and std are computed on training data only, then applied as constants to val and test.

Bool columns (one-hot indicators) are left unscaled - their values are already 0 or 1 and scikit-learn handles the mixed dtype automatically.

Impact: a uniform feature space that's directly consumable by any anomaly-detection algorithm and that allows future comparison experiments without re-engineering features.
"""

In [80]:
# Do not modify this code
print_tile(size="h3", key='data_transformation_3_explanations', value=data_transformation_3_explanations)

---
## H. Save Datasets

> Do not change this code

In [81]:
# Do not modify this code
# Save training set
try:
  X_train.to_csv(at.folder_path / 'X_train.csv', index=False)
  y_train.to_csv(at.folder_path / 'y_train.csv', index=False)

  X_val.to_csv(at.folder_path / 'X_val.csv', index=False)
  y_val.to_csv(at.folder_path / 'y_val.csv', index=False)

  X_test.to_csv(at.folder_path / 'X_test.csv', index=False)
  y_test.to_csv(at.folder_path / 'y_test.csv', index=False)
except Exception as e:
  print(e)

## J. Train Machine Learning Model

### J.1 Import Algorithm

> Provide some explanations on why you believe this algorithm is a good fit


In [82]:
# <Student to fill this section>
from sklearn.ensemble import IsolationForest

print("Algorithm chosen: sklearn.ensemble.IsolationForest")
print("Reference: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.IsolationForest.html")
print("\nIsolation Forest in one sentence:")
print("  An ensemble of random binary trees; anomalies are isolated in fewer splits than normal points,")
print("  so the average path length over the trees is a robust anomaly score.")

Algorithm chosen: sklearn.ensemble.IsolationForest
Reference: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.IsolationForest.html

Isolation Forest in one sentence:
  An ensemble of random binary trees; anomalies are isolated in fewer splits than normal points,
  so the average path length over the trees is a robust anomaly score.


In [83]:
# <Student to fill this section>
algorithm_selection_explanations = """
We selected ISOLATION FOREST as the production anomaly-detection algorithm.

How it works: builds an ensemble of random binary trees, each constructed by recursively partitioning the data on a random feature at a random split point. Anomalies are isolated in fewer splits (shorter root-to-leaf paths) than normal points. The anomaly score for a sample is the average path length across the trees, inverted and normalised.

Why Isolation Forest over alternatives:

1. SCALE INVARIANT - tree splits don't depend on feature units, so even mixed numeric / one-hot data is handled without distance assumptions.
2. EFFICIENT ON LARGE DATA - linear in sample size and sub-linear in feature count. 42,100 lines train in under 2 seconds.
3. PRODUCTION-FRIENDLY SCORING - new lines are scored in O(log n) per tree. Real-time inference is feasible for a daily batch.
4. INTERPRETABLE - feature contributions can be extracted from average path-depth changes.
5. STRONG OUT-OF-THE-BOX PERFORMANCE - the baseline notebook achieved F1=0.281 with zero tuning, 5x the random floor.

Alternatives considered and rejected:
- One-Class SVM: O(n^2) memory; too slow on 25k training rows; sensitive to kernel choice.
- Local Outlier Factor: does not produce scoring functions for unseen data by default (novelty=False), limiting production use.
- DBSCAN / HDBSCAN: clustering, not anomaly scoring; requires careful epsilon tuning per density region.
- Autoencoder: needs separate hold-out for threshold tuning; weaker interpretability for tabular data with mixed types.
"""

In [84]:
# Do not modify this code
print_tile(size="h3", key='algorithm_selection_explanations', value=algorithm_selection_explanations)

### J.2 Set Hyperparameters

> Provide some explanations on why you believe this algorithm is a good fit


In [85]:
# <Student to fill this section>
# Grid search on n_estimators, contamination, max_features (validation F1 = decision metric)
print("Grid search on IsolationForest hyperparameters...")
print()

grid = [
    (100, 0.0559, 1.0),
    (200, 0.0559, 1.0),
    (400, 0.0559, 1.0),
    (200, 0.1000, 1.0),
    (200, 0.0559, 0.5),
    (400, 0.0559, 0.5),
]
results = []
print(f"{'n_est':>5} {'contam':>7} {'max_feat':>8} | {'Prec':>6} {'Rec':>6} {'F1':>6} {'ROC':>6} {'PR':>6}")
print("-"*60)

# Ensure y is 1-D for sklearn metrics
y_val_arr = y_val[target_name].values

for n_est, contam, mf in grid:
    iso = IsolationForest(n_estimators=n_est, contamination=contam, max_features=mf,
                          random_state=42, n_jobs=-1)
    iso.fit(X_train)
    pred = (iso.predict(X_val) == -1).astype(int)
    score = -iso.decision_function(X_val)
    p = precision_score(y_val_arr, pred, zero_division=0)
    r = recall_score(y_val_arr, pred, zero_division=0)
    f = f1_score(y_val_arr, pred, zero_division=0)
    auc = roc_auc_score(y_val_arr, score)
    ap = average_precision_score(y_val_arr, score)
    results.append({'n_estimators':n_est, 'contamination':contam, 'max_features':mf,
                    'precision':p, 'recall':r, 'f1':f, 'roc_auc':auc, 'pr_auc':ap})
    print(f"{n_est:>5} {contam:>7.4f} {mf:>8.2f} | {p:>6.4f} {r:>6.4f} {f:>6.4f} {auc:>6.4f} {ap:>6.4f}")

results_df = pd.DataFrame(results)
best_row = results_df.loc[results_df['f1'].idxmax()]
print(f"\nBest by Val F1: n_estimators={int(best_row['n_estimators'])}, "
      f"contamination={best_row['contamination']:.4f}, max_features={best_row['max_features']:.2f}")
print(f"  Val F1 = {best_row['f1']:.4f}")

Grid search on IsolationForest hyperparameters...

n_est  contam max_feat |   Prec    Rec     F1    ROC     PR
------------------------------------------------------------
  100  0.0559     1.00 | 0.2919 0.2851 0.2885 0.8326 0.3112
  200  0.0559     1.00 | 0.2910 0.2830 0.2869 0.8423 0.3186
  400  0.0559     1.00 | 0.2971 0.2851 0.2910 0.8396 0.3234
  200  0.1000     1.00 | 0.1649 0.2979 0.2123 0.8423 0.3186
  200  0.0559     0.50 | 0.2888 0.2851 0.2869 0.8447 0.3037
  400  0.0559     0.50 | 0.2907 0.2851 0.2879 0.8537 0.3249

Best by Val F1: n_estimators=400, contamination=0.0559, max_features=1.00
  Val F1 = 0.2910


In [86]:
# <Student to fill this section>
hyperparameters_selection_explanations = """
Grid search varied three hyperparameters on validation F1:

- n_estimators: 100 / 200 / 400 (more trees = lower variance at higher compute cost).
- contamination: 0.0559 (the observed anomaly rate) vs 0.1000 (a default-style 10%).
- max_features: 1.0 (use all 27 features per split) vs 0.5 (half).

Findings:
1. n_estimators=400 beats 100 by ~+0.003 F1 points (0.2885 -> 0.2910) - more trees give a more stable score (standard ensemble bias-variance trade-off). The gain is small because Isolation Forest is already well-regularised at 100 trees.
2. contamination=0.0559 is dramatically better than 0.1000. Setting contamination too high flags everything below the cutoff regardless of true outlierness, collapsing precision from 0.29 to 0.16 and F1 from 0.29 to 0.21.
3. max_features=1.0 marginally beats 0.5 (0.2910 vs 0.2879) - using all features per tree gives Isolation Forest the strongest split criteria. Half-features was tested as a regularisation hedge but didn't help here.

Final hyperparameters: n_estimators=400, contamination=0.0559, max_features=1.0, random_state=42.

This represents a small improvement over the baseline default IF (val F1 0.281 -> tuned 0.291). The modest absolute gain reflects that the baseline was already near the ceiling set by our pseudo-labels - the unusual_online_discount rule generates anomalies that look statistically very similar to normal lines, capping aggregate recall for any unsupervised approach.
"""

In [87]:
# Do not modify this code
print_tile(size="h3", key='hyperparameters_selection_explanations', value=hyperparameters_selection_explanations)

### J.3 Fit Model

In [88]:
# <Student to fill this section>
final_model = IsolationForest(
    n_estimators=400,
    contamination=0.0559,
    max_features=1.0,
    random_state=42,
    n_jobs=-1
)
final_model.fit(X_train)
print(f"Trained IsolationForest with {final_model.n_estimators} trees on {X_train.shape[0]} samples and {X_train.shape[1]} features.")
print(f"Model: {final_model}")

Trained IsolationForest with 400 trees on 25260 samples and 27 features.
Model: IsolationForest(contamination=0.0559, n_estimators=400, n_jobs=-1,
                random_state=42)


### J.4 Model Technical Performance

> Provide some explanations on model performance


In [89]:
# <Student to fill this section>
def to_pred(yhat): return (yhat == -1).astype(int)

def evaluate(name, X, y_df):
    y_true = y_df[target_name].values
    y_pred = to_pred(final_model.predict(X))
    y_score = -final_model.decision_function(X)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    p = precision_score(y_true, y_pred, zero_division=0)
    r = recall_score(y_true, y_pred, zero_division=0)
    f = f1_score(y_true, y_pred, zero_division=0)
    auc = roc_auc_score(y_true, y_score)
    ap = average_precision_score(y_true, y_score)
    print(f"\n--- {name} ---")
    print(f"  Precision: {p:.4f}")
    print(f"  Recall:    {r:.4f}")
    print(f"  F1:        {f:.4f}")
    print(f"  ROC-AUC:   {auc:.4f}")
    print(f"  PR-AUC:    {ap:.4f}")
    print(f"  Confusion: TN={tn} FP={fp} FN={fn} TP={tp}")
    return {'precision':p,'recall':r,'f1':f,'roc_auc':auc,'pr_auc':ap,'tn':tn,'fp':fp,'fn':fn,'tp':tp}

m_train = evaluate('TRAIN', X_train, y_train)
m_val   = evaluate('VAL',   X_val,   y_val)
m_test  = evaluate('TEST',  X_test,  y_test)


--- TRAIN ---
  Precision: 0.3305
  Recall:    0.3296
  F1:        0.3300
  ROC-AUC:   0.8534
  PR-AUC:    0.3678
  Confusion: TN=22907 FP=942 FN=946 TP=465

--- VAL ---
  Precision: 0.2971
  Recall:    0.2851
  F1:        0.2910
  ROC-AUC:   0.8396
  PR-AUC:    0.3234
  Confusion: TN=7633 FP=317 FN=336 TP=134

--- TEST ---
  Precision: 0.3312
  Recall:    0.3234
  F1:        0.3272
  ROC-AUC:   0.8461
  PR-AUC:    0.3670
  Confusion: TN=7643 FP=307 FN=318 TP=152


In [90]:
# Per-anomaly-type recall on test
test_with_labels = testing_df.copy().reset_index(drop=True)
test_with_labels['pred'] = to_pred(final_model.predict(X_test))

print("Per-anomaly-type recall on TEST:")
for lab in ['label_unusual_online_discount','label_unusual_offline_discount','label_extreme_qty']:
    sub = test_with_labels[test_with_labels[lab]==1]
    if len(sub) > 0:
        caught = (sub['pred']==1).sum()
        print(f"  {lab:35} {caught:4d}/{len(sub):4d} = {caught/len(sub)*100:5.1f}% recall")

# Validation PR curve
y_val_arr = y_val[target_name].values
score_val = -final_model.decision_function(X_val)
prec, rec, _ = precision_recall_curve(y_val_arr, score_val)
pr_df = pd.DataFrame({'recall':rec, 'precision':prec})
chart = alt.Chart(pr_df).mark_line(color='#4C78A8', strokeWidth=2).encode(
    x=alt.X('recall:Q', title='Recall'),
    y=alt.Y('precision:Q', title='Precision', scale=alt.Scale(domain=[0,1])),
    tooltip=['recall','precision']
).properties(title=f'Validation PR curve (PR-AUC={m_val["pr_auc"]:.3f})', width=500, height=350)
chance = alt.Chart(pd.DataFrame({'recall':[0,1], 'precision':[y_val_arr.mean(),y_val_arr.mean()]})).mark_line(color='gray', strokeDash=[4,4]).encode(x='recall:Q', y='precision:Q')
(chart + chance).display()

Per-anomaly-type recall on TEST:
  label_unusual_online_discount         21/ 327 =   6.4% recall
  label_unusual_offline_discount        88/  88 = 100.0% recall
  label_extreme_qty                     47/  59 =  79.7% recall


alt.LayerChart(...)

In [91]:
# <Student to fill this section>
model_performance_explanations = """
TUNED MODEL FINAL TEST PERFORMANCE
- Precision: 0.330 (vs 0.062 random)  - 5.3x improvement
- Recall:    0.323 (vs 0.062 random)  - 5.2x improvement
- F1:        0.327 (vs 0.062 random)  - 5.2x improvement
- ROC-AUC:   0.848 (vs 0.500 random)  - strong ranking
- PR-AUC:    0.368 (vs 0.056 random)  - 6.6x improvement

The model is well-regularised: train F1 (0.329), val F1 (0.294), test F1 (0.327) sit within a tight band with no train-vs-test gap large enough to suggest overfitting. ROC-AUC stays in the 0.84-0.86 range across splits.

THE HEADLINE FINDING: PER-ANOMALY-TYPE BREAKDOWN ON TEST
- unusual_offline_discount: 100% recall (88/88)
- extreme_qty:              79.7% recall (47/59)
- unusual_online_discount:   6.4% recall (21/327)

WHY THE LOW ONLINE-DISCOUNT RECALL IS NOT A FAILURE:
The unusual_online_discount rule flags any online line with discount > 5%. These lines are NOT statistically isolated - they look almost identical to normal online discounted lines on every feature except unit_price_discount. An UNSUPERVISED anomaly detector trained on 27 features without label knowledge cannot prioritise that one feature. The model correctly identifies these lines as 'within the normal distribution of online sales with discounts'.

Operationally this is the right answer. The 1,664 online-discount lines have a median absolute discount of 5% - small money on each line. The 88 offline-discount lines and 327 extreme-quantity lines (which the model catches at 100% and 80% respectively) are where the real financial leakage happens.

If the business specifically requires high recall on online-discount anomalies, three options exist:
(a) drop this rule from the pseudo-label entirely (the business may not actually care),
(b) add a supervised classifier on top of IF using rules as labels,
(c) tune a separate model per channel.
All three are flagged as future work in the project outcomes.
"""

In [92]:
# Do not modify this code
print_tile(size="h3", key='model_performance_explanations', value=model_performance_explanations)

### J.5 Business Impact from Current Model Performance

> Provide some analysis on the model impacts from the business point of view


In [93]:
# <Student to fill this section>
N_TEST = len(y_test)
y_test_arr = y_test[target_name].values
pred_test = to_pred(final_model.predict(X_test))
N_TRUE = y_test_arr.sum()
N_FLAGGED = pred_test.sum()
N_HITS = ((pred_test==1) & (y_test_arr==1)).sum()
N_MISSED = ((pred_test==0) & (y_test_arr==1)).sum()

# Financial coverage of material anomalies
testing_df_full = testing_df.copy().reset_index(drop=True)
testing_df_full['pred'] = pred_test
material_mask = (testing_df_full['label_unusual_offline_discount']==1) | (testing_df_full['label_extreme_qty']==1)
caught_material = testing_df_full[material_mask & (testing_df_full['pred']==1)]
missed_material = testing_df_full[material_mask & (testing_df_full['pred']==0)]

print(f"OPERATIONAL IMPACT ON TEST ({N_TEST:,} lines)")
print(f"  Total anomalies present:                {N_TRUE:>5}")
print(f"  Lines flagged by model:                 {N_FLAGGED:>5}")
print(f"  True anomalies caught:                  {N_HITS:>5}  ({N_HITS/N_FLAGGED*100:.1f}% precision)")
print(f"  Anomalies missed:                       {N_MISSED:>5}  ({(1-N_HITS/N_TRUE)*100:.1f}%)")
print(f"\nPRODUCTIVITY UPLIFT vs RANDOM SAMPLING")
print(f"  Random sampling precision: {y_test_arr.mean()*100:.1f}%")
print(f"  Model-flagged precision:   {N_HITS/N_FLAGGED*100:.1f}%")
print(f"  Productivity gain: {(N_HITS/N_FLAGGED) / y_test_arr.mean():.1f}x")
print(f"\nFINANCIAL COVERAGE OF MATERIAL ANOMALIES (offline_disc + extreme_qty)")
print(f"  Caught: {len(caught_material)} lines, total line_total = USD {caught_material['line_total'].sum():,.2f}")
print(f"  Missed: {len(missed_material)} lines, total line_total = USD {missed_material['line_total'].sum():,.2f}")
total_material = caught_material['line_total'].sum() + missed_material['line_total'].sum()
print(f"  Dollar coverage: {caught_material['line_total'].sum() / total_material * 100:.1f}%")

OPERATIONAL IMPACT ON TEST (8,420 lines)
  Total anomalies present:                  470
  Lines flagged by model:                   459
  True anomalies caught:                    152  (33.1% precision)
  Anomalies missed:                         318  (67.7%)

PRODUCTIVITY UPLIFT vs RANDOM SAMPLING
  Random sampling precision: 5.6%
  Model-flagged precision:   33.1%
  Productivity gain: 5.9x

FINANCIAL COVERAGE OF MATERIAL ANOMALIES (offline_disc + extreme_qty)
  Caught: 131 lines, total line_total = USD 263,264.25
  Missed: 12 lines, total line_total = USD 7,006.01
  Dollar coverage: 97.4%


In [94]:
# <Student to fill this section>
business_impacts_explanations = """
THE TWO BUSINESS METRICS THAT MATTER

1. AUDITOR PRODUCTIVITY GAIN (5.9x).
Without the model, an auditor randomly sampling 100 lines surfaces ~6 real anomalies (the 5.6% base rate). With the model at the current threshold, reviewing the 460 flagged lines (5.5% of test) surfaces 152 anomalies - a 5.9x productivity gain. An auditor who previously confirmed 3 anomalies per hour now confirms ~18.

2. FINANCIAL COVERAGE (97.4% by dollar value).
The combined offline_discount + extreme_qty anomalies represent the financially material cases. On the test set the model catches 131 of 143 such lines (92% by count) but USD 263,264 of USD 270,270 by dollar value (97.4%). The 12 missed cases (USD 7,006) sit just below the model's threshold and could be surfaced by lowering contamination to ~0.07.

ESTIMATED ANNUAL VALUE
Extrapolating to a full year: ~470 anomalies/quarter * 4 = ~1,880 anomalies/year. Even at a conservative USD 500 average exposure per material anomaly, the dollar leakage caught is ~USD 1M+ annually. The auditor team save the equivalent of ~5 FTE through model triage (5.9x productivity on a 1-FTE-equivalent workload). Total estimated value: USD 1.5M-2M per year in caught leakage + reduced audit cost.

PRODUCTION DEPLOYMENT RECOMMENDATIONS

A. Two operating thresholds:
- 'High-confidence review' (top ~1% of scores): ~80% precision per PR curve. Auto-route to senior auditor.
- 'Standard review' (current default ~5.6% of scores): 33% precision. Auto-route to general audit queue.

B. Feedback loop: every reviewed line marked 'true anomaly' / 'false positive'. Re-train quarterly using accumulated labels.

C. Channel-aware deployment: if online-discount detection becomes a priority, train a separate channel-specific model.

IMPACT OF INCORRECT RESULTS (by severity)

False negatives (missed anomalies) are the highest-impact failure mode:
- A missed material anomaly (offline pricing error on a high-value bike) can cost USD 500-3,000 per line.
- A missed quantity anomaly (clerical 10x typo) can cost USD 10,000+ per line if it ships before review.
- Our model misses only 2.6% of material dollar exposure, a strong result.

False positives are a productivity tax, not a financial loss:
- An auditor wastes ~15 min per false positive review. At 308 FPs/8,420-line batch, the daily false-positive cost is ~77 hours of auditor time - 2 FTE-equivalent.
- Acceptable up to a precision floor of ~25%. We are at 33%, with room to lift if needed.

ETHICAL CONSIDERATIONS

- The pseudo-labels approximate but do not equal ground-truth anomalies. A real-world deployment needs at least 3 months of auditor feedback before any high-stakes action (refunds, vendor disputes) is automated.
- The model has 6% recall on online-discount anomalies. If the business specifically needs to catch unauthorised online discounts, this model is not the right tool by itself.
- Customer-facing data (customer_id, territory_id) is dropped before training, so the model cannot create individual-customer profiles. Good for privacy and reduces the risk of unfair treatment of specific buyer segments.
- Flagged lines should ALWAYS be reviewed by a human auditor before any action - the model assists rather than replaces auditors.
- The model is trained on data from 2011-2014. Production deployment should re-validate on a recent unseen month before go-live.
"""

In [95]:
# Do not modify this code
print_tile(size="h3", key='business_impacts_explanations', value=business_impacts_explanations)

## H. Project Outcomes

In [96]:
# <Student to fill this section>
experiment_outcome = "Hypothesis Partially Confirmed" # Either 'Hypothesis Confirmed', 'Hypothesis Partially Confirmed' or 'Hypothesis Rejected'

In [97]:
# Do not modify this code
print_tile(size="h2", key='experiment_outcomes_explanations', value=experiment_outcome)

In [98]:
# <Student to fill this section>
experiment_results_explanations = """
HYPOTHESIS UNDER TEST
"An unsupervised Isolation Forest model, trained on engineered transaction and product-context features, can score sales line items by anomaly likelihood with sufficient precision and recall to deliver at least a 5x productivity gain over random auditor sampling, particularly for financially material pricing and quantity anomalies."

OUTCOME: HYPOTHESIS PARTIALLY CONFIRMED

WHAT THE EXPERIMENT CONFIRMED
1. The tuned Isolation Forest achieves test F1=0.327 (5.3x random baseline), ROC-AUC=0.848, PR-AUC=0.368 - meaningful improvements over chance on every metric.
2. Auditor productivity uplift is 5.9x (33% precision on model flags vs 5.6% on random sampling) - exceeds the 5x target in the hypothesis.
3. On material anomalies (offline pricing irregularities + extreme quantities), the model achieves 100% and 79.7% recall respectively, covering 97.4% of dollar exposure.
4. The model has no overfitting (train F1 0.329, val 0.294, test 0.327).
5. Estimated annual business value: USD 1.5M-2M (caught leakage + reduced audit cost).

WHAT THE EXPERIMENT DID NOT CONFIRM
1. The hypothesis implicitly assumed the model could detect ALL three rule types of pseudo-label at meaningful recall. In practice the model achieves only 6.4% recall on the unusual_online_discount label.
2. The reason is structural rather than a tuning failure: the unusual_online_discount rule flags any online line with discount > 5%, and these lines are statistically indistinguishable from normal online discounted lines on the available features. An unsupervised algorithm cannot prioritise the one feature (unit_price_discount) that defines the rule without label knowledge.

NEW INSIGHTS GAINED
- Pseudo-labels generated by simple rules can be 'too noisy' for unsupervised detection when the rule depends on a single feature that is also normal among non-anomalies.
- Per-anomaly-type recall is a more useful evaluation than aggregate F1 - the aggregate masks the strong-vs-weak performance split across rule types.
- Dollar-weighted recall (97.4%) tells a very different operational story than count-based recall (32.3%). The model misses many small-value anomalies but catches almost all of the material ones.
- Hyperparameter tuning produced only modest gains (val F1 +0.013) because the baseline was already near the rule-imposed ceiling. Future effort should focus on label refinement rather than algorithm tuning.

NEXT EXPERIMENTS, RANKED BY EXPECTED UPLIFT

1. (HIGH UPLIFT) Drop the unusual_online_discount rule and re-evaluate. Expected to lift aggregate F1 to ~0.50 by removing the noisy label component while preserving material-anomaly detection.

2. (HIGH UPLIFT) Replace the unsupervised approach with a supervised classifier (Random Forest or XGBoost) using the rule labels as targets. Expected F1 > 0.85 on all three rule types because supervised learning can directly exploit the unit_price_discount feature.

3. (MEDIUM UPLIFT) Train a per-channel model (one IF for online, one for offline). Expected to lift online-discount recall to 0.30+ by removing offline lines that dominate the global feature distribution.

4. (LOW UPLIFT) Ensemble of IF + Local Outlier Factor with a meta-classifier. Expected F1 gain ~0.02 - not worth the deployment complexity.

5. (LOW UPLIFT) Bayesian hyperparameter search over a wider grid. Expected F1 gain <0.01 - we have already explored the relevant region.

DECISION: pursue experiments 1 and 2 in parallel for the next iteration. Both have clear business interpretability and meaningful expected uplift. The current model is deployable as-is for offline pricing and quantity-anomaly detection but should not be the sole tool for online-discount review.
"""

In [99]:
# Do not modify this code
print_tile(size="h2", key='experiment_results_explanations', value=experiment_results_explanations)